# PharmaCast

Forecasting weekly pharmaceutical demand from point of sale data. Side project, but I tried to keep it honest: every model has to beat a naive baseline under walk forward cross validation, otherwise it does not make the cut.

Data: salesdaily.csv, daily sales for 8 ATC drug categories, 2014 to 2019.


## Setup

In [ ]:
!pip install -q prophet pmdarima lightgbm statsmodels
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
print("Ready")

## Load + data quality
First thing is checking the data is actually clean before trusting anything.

In [ ]:
df = pd.read_csv('salesdaily.csv')
df['date'] = pd.to_datetime(df['datum'], format='%m/%d/%Y')
df = df.sort_values('date').reset_index(drop=True)
cats = ['M01AB','M01AE','N02BA','N02BE','N05B','N05C','R03','R06']
df['total'] = df[cats].sum(axis=1)
print(df.shape, df['date'].min().date(), "->", df['date'].max().date())
df.head()

In [ ]:
full = pd.date_range(df['date'].min(), df['date'].max(), freq='D')
print("Calendar days:", len(full), "| Rows:", len(df), "| Missing dates:", len(full)-len(df))
print("\nNulls:\n", df[cats].isna().sum())
print("\nDescribe:\n", df[cats].describe().T[['mean','std','min','max']].round(1))
print("\nZero-sales days per category:\n", (df[cats]==0).sum())

## EDA
Monthly view to see the trend and seasonality. N05C is clearly sparse, noting that for later.

In [ ]:
m = df.set_index('date')[cats].resample('M').sum()
m['total'] = m.sum(axis=1)
fig, ax = plt.subplots(2,1, figsize=(13,8))
m['total'].plot(ax=ax[0], title='Total monthly demand'); ax[0].grid(alpha=.3)
m[cats].plot(ax=ax[1], title='Monthly demand by category', lw=1); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

## Monthly modeling
Start monthly. Build a clean series, drop the incomplete last month, hold out the last 6 months.

In [ ]:
m = df.set_index('date')[cats].resample('M').sum()
m['total'] = m.sum(axis=1)
m = m.iloc[:-1]                     # drop incomplete Oct-2019
y = m['total'].astype(float)
TEST = 6
train, test = y[:-TEST], y[-TEST:]
print("Train:", train.index.min().date(), "->", train.index.max().date(), "| n =", len(train))
print("Test :", test.index.min().date(),  "->", test.index.max().date(),  "| n =", len(test))

STL decomposition to confirm there is real seasonality worth modeling.

In [ ]:
from statsmodels.tsa.seasonal import STL
res = STL(y, period=12, robust=True).fit()
res.plot(); plt.tight_layout(); plt.show()
print("Seasonal strength:", round(1 - res.resid.var()/(res.seasonal+res.resid).var(), 3))

Baselines first. These are the numbers everything else has to beat.

In [ ]:
def mape(a,f): a,f=np.array(a),np.array(f); return np.mean(np.abs((a-f)/a))*100
def mae(a,f):  return np.mean(np.abs(np.array(a)-np.array(f)))
snaive = y.shift(12).iloc[-TEST:]
naive  = pd.Series(train.iloc[-1], index=test.index)
print(f"Naive         -> MAE {mae(test,naive):.1f} | MAPE {mape(test,naive):.1f}%")
print(f"Seasonal-Naive-> MAE {mae(test,snaive):.1f} | MAPE {mape(test,snaive):.1f}%")

SARIMAX with auto order.

In [ ]:
import pmdarima as pm
auto = pm.auto_arima(train, seasonal=True, m=12, stepwise=True,
                     suppress_warnings=True, trace=True)
print(auto.summary())
fc = pd.Series(auto.predict(n_periods=TEST), index=test.index)
print(f"\nSARIMAX -> MAE {mae(test,fc):.1f} | MAPE {mape(test,fc):.1f}%")
plt.figure(figsize=(12,5))
train.plot(label='train'); test.plot(label='actual', lw=2); fc.plot(label='SARIMAX', ls='--')
plt.legend(); plt.title('SARIMAX vs actual (6-mo holdout)'); plt.grid(alpha=.3); plt.show()

SARIMAX (11.8% rolling) lost to seasonal naive (9.0%). Only 63 monthly points, seasonal terms insignificant, auto picked zero differencing. The honest read is that monthly aggregation just does not give the models enough signal. So I switch to weekly.

## Weekly modeling + rolling-origin CV
Reusable walk forward backtest so every model is judged the same way.

In [ ]:
w = df.set_index('date')[cats].resample('W').sum()
w['total'] = w.sum(axis=1)
w = w.iloc[:-1]
yw = w['total'].astype(float)
print("Weekly obs:", len(yw), "|", yw.index.min().date(), "->", yw.index.max().date())

H = 12
def rolling_bt(predict_fn, y, H=12, folds=6, step=4):
    scores=[]
    for k in range(folds):
        cut=len(y)-H-k*step
        if cut < 60: break
        tr, te = y.iloc[:cut], y.iloc[cut:cut+H]
        scores.append(mape(te.values, predict_fn(tr, te, H)))
    return np.mean(scores), [round(s,1) for s in scores]

def snaive_fn(tr, te, H):
    hist=pd.concat([tr,te]); return hist.shift(52).loc[te.index].values
mu,sc=rolling_bt(snaive_fn, yw); print(f"Seasonal-Naive (weekly) -> mean MAPE {mu:.1f}%  {sc}")

Prophet.

In [ ]:
from prophet import Prophet
def prophet_fn(tr, te, H):
    d=tr.reset_index(); d.columns=['ds','y']
    mdl=Prophet(yearly_seasonality=True, weekly_seasonality=False,
                daily_seasonality=False, seasonality_mode='multiplicative')
    mdl.fit(d)
    fut=mdl.make_future_dataframe(periods=H, freq='W')
    return mdl.predict(fut).set_index('ds')['yhat'].loc[te.index].values
mu,sc=rolling_bt(prophet_fn, yw); print(f"Prophet -> mean MAPE {mu:.1f}%  {sc}")

LightGBM with lag, rolling and calendar features, forecast recursively over the horizon.

In [ ]:
import lightgbm as lgb
def make_feats(s):
    d = pd.DataFrame({'y': s})
    for L in [1,2,3,4,8,12,52]:
        d[f'lag{L}'] = d['y'].shift(L)
    d['roll4']  = d['y'].shift(1).rolling(4).mean()
    d['roll12'] = d['y'].shift(1).rolling(12).mean()
    woy = d.index.isocalendar().week.astype(int)
    d['woy_sin'] = np.sin(2*np.pi*woy/52); d['woy_cos'] = np.cos(2*np.pi*woy/52)
    d['month']   = d.index.month
    return d
FEATS = ['lag1','lag2','lag3','lag4','lag8','lag12','lag52',
         'roll4','roll12','woy_sin','woy_cos','month']
PARAMS = dict(n_estimators=400, learning_rate=0.03, num_leaves=31,
              min_child_samples=10, subsample=0.8, colsample_bytree=0.8,
              random_state=42, verbose=-1)
def lgbm_fn(tr, te, H):
    hist = tr.copy()
    model = lgb.LGBMRegressor(**PARAMS).fit(*(lambda d:(d[FEATS],d['y']))(make_feats(hist).dropna()))
    preds=[]
    for _ in range(H):
        nxt = hist.index[-1] + pd.Timedelta(weeks=1)
        ext = pd.concat([hist, pd.Series([np.nan], index=[nxt])])
        f = make_feats(ext).loc[[nxt], FEATS]
        yhat = float(model.predict(f)[0]); preds.append(yhat)
        hist = pd.concat([hist, pd.Series([yhat], index=[nxt])])
    return np.array(preds)
mu, sc = rolling_bt(lgbm_fn, yw); print(f"LightGBM -> mean MAPE {mu:.1f}%  {sc}")

Leaderboard: Prophet 12.2%, Seasonal-Naive 14.3%, LightGBM 15.5%. Prophet is the only learned model that beats its baseline. LightGBM loses because the recursive forecast compounds its own errors and 300 points is not enough for it. Prophet wins.

## Feature importance (LightGBM)
Even though LightGBM lost, the importances are a nice sanity check on what drives demand.

In [ ]:
d = make_feats(yw).dropna()
imp_model = lgb.LGBMRegressor(**PARAMS).fit(d[FEATS], d['y'])
imp = pd.Series(imp_model.feature_importances_, index=FEATS).sort_values()
plt.figure(figsize=(8,5))
imp.plot.barh(color='#4f46e5'); plt.title('LightGBM feature importance (gain splits)')
plt.tight_layout(); plt.show()
print(imp[::-1])

## Final Prophet fit + residual diagnostics
Refit on all weekly data, then check residuals. Good residuals should look like noise with no leftover autocorrelation.

In [ ]:
d = yw.reset_index(); d.columns = ['ds','y']
final = Prophet(yearly_seasonality=True, weekly_seasonality=False,
                daily_seasonality=False, seasonality_mode='multiplicative',
                interval_width=0.80)
final.fit(d)
insample = final.predict(d[['ds']]).set_index('ds')['yhat']
resid = yw - insample.values

fig, ax = plt.subplots(1,2, figsize=(13,4))
ax[0].scatter(insample.values, resid, s=10, alpha=.6); ax[0].axhline(0, color='r', lw=1)
ax[0].set_title('Residuals vs fitted'); ax[0].set_xlabel('fitted'); ax[0].set_ylabel('residual')
ax[1].hist(resid, bins=30, color='#0ea5e9'); ax[1].set_title('Residual distribution')
plt.tight_layout(); plt.show()

from statsmodels.stats.diagnostic import acorr_ljungbox
lb = acorr_ljungbox(resid, lags=[12], return_df=True)
print(lb)
print("Residual mean:", round(resid.mean(),2), "| std:", round(resid.std(),2))

## Export champion forecast for the dashboard

In [ ]:
import json
fut = final.make_future_dataframe(periods=12, freq='W')
fcst = final.predict(fut)
export = {
  "granularity":"weekly","champion":"Prophet (multiplicative yearly seasonality)","horizon_weeks":12,
  "leaderboard":[
     {"model":"Naive","gran":"monthly","mape":27.7},
     {"model":"Seasonal-Naive","gran":"monthly","mape":9.0},
     {"model":"SARIMAX (auto)","gran":"monthly","mape":11.8},
     {"model":"Seasonal-Naive","gran":"weekly","mape":14.3},
     {"model":"LightGBM (recursive)","gran":"weekly","mape":15.5},
     {"model":"Prophet","gran":"weekly","mape":12.2}],
  "weeks":[d.strftime('%Y-%m-%d') for d in yw.index],
  "actual":[round(float(v),1) for v in yw.values],
  "fc_weeks":[d.strftime('%Y-%m-%d') for d in fcst['ds'].iloc[-12:]],
  "fc_mean":[round(float(v),1) for v in fcst['yhat'].iloc[-12:]],
  "fc_lo":[round(float(v),1) for v in fcst['yhat_lower'].iloc[-12:]],
  "fc_hi":[round(float(v),1) for v in fcst['yhat_upper'].iloc[-12:]]}
with open('pharmacast_forecast.json','w') as f: json.dump(export, f)
print("Exported. Champion: Prophet @ 12.2% MAPE")

## Intermittent demand side quest: N05C

N05C sells nothing 68% of days, which is intermittent demand. The usual advice is Croston's method, but I wanted to actually test it. Turns out the intermittency depends on granularity, and weekly aggregation mostly removes it. Where the zeros really live (daily), I score with MASE since MAPE breaks when actuals are zero.

In [ ]:
def croston(y, alpha=0.1):
    y=np.asarray(y,float); z=q=None; p=1; f=np.full(len(y),np.nan)
    for t,dd in enumerate(y):
        if z is None:
            if dd>0: z,q=dd,1.0
            continue
        if dd>0: z=alpha*dd+(1-alpha)*z; q=alpha*p+(1-alpha)*q; p=1
        else: p+=1
        f[t]=z/q
    return f

yd = df.set_index('date')['N05C'].astype(float)
print("Daily non-zero:", f"{100*(yd>0).mean():.0f}%", "| mean:", round(yd.mean(),2))

def mase(train, actual, fcst):
    denom = np.mean(np.abs(np.diff(train.values)))
    return np.mean(np.abs(actual.values - fcst)) / denom
def bt_mase(predict_fn, y, H=28, folds=6, step=28):
    out=[]
    for k in range(folds):
        cut=len(y)-H-k*step
        if cut < 400: break
        tr, te = y.iloc[:cut], y.iloc[cut:cut+H]
        out.append(mase(tr, te, predict_fn(tr, te, H)))
    return np.mean(out), [round(s,3) for s in out]
def f_mean(tr,te,H):  return np.full(H, tr.mean())
def f_naive(tr,te,H): return np.full(H, tr.iloc[-1])
def f_croston(tr,te,H):
    f=croston(tr.values,0.1); lvl=f[-1]
    return np.full(H, lvl if not np.isnan(lvl) else tr.mean())
for name, fn in [("Naive (last)",f_naive),("Mean",f_mean),("Croston",f_croston)]:
    mu, sc = bt_mase(fn, yd); print(f"{name:14s} -> MASE {mu:.3f}  {sc}")

Croston's came last (MASE 1.059), losing to naive (0.816). N05C is lumpy but still fairly frequent, not the sparse spiky demand Croston's was built for. Lesson: do not reach for a specialized model on reflex, validate it. For N05C the better move is to forecast it weekly as part of the total, where Prophet already works.